In [ ]:
import pandas as pd
import plotly.express as px
from datetime import datetime
import pytz
import hashlib

print("🌍 Global App Installs by Category")

# --------------------------------------------------
# Time restriction function (6 PM – 8 PM IST)
# --------------------------------------------------
def is_allowed_time():
    ist = pytz.timezone("Asia/Kolkata")
    now = datetime.now(ist)
    return 18 <= now.hour < 20

# --------------------------------------------------
# Block dashboard outside allowed time
# --------------------------------------------------
if not is_allowed_time():
    print("⛔ Dashboard available only between 6 PM and 8 PM IST")
else:

    # --------------------------------------------------
    # Load Dataset
    # --------------------------------------------------
    df = pd.read_csv("Play Store Data.csv")

    # Normalize column names
    df.columns = df.columns.str.strip().str.lower()

    # --------------------------------------------------
    # Clean installs column
    # --------------------------------------------------
    df["installs"] = (
        df["installs"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False)
    )

    df = df[df["installs"].str.isnumeric()]
    df["installs"] = df["installs"].astype(int)

    # --------------------------------------------------
    # Remove categories starting with A, C, G, S
    # --------------------------------------------------
    df = df[~df["category"].str.startswith(("A", "C", "G", "S"))]

    # --------------------------------------------------
    # Create deterministic COUNTRY column
    # --------------------------------------------------
    countries = [
        "India", "United States", "Germany",
        "Brazil", "United Kingdom", "Canada",
        "Australia", "France", "Japan"
    ]

    def assign_country(app_name):
        hash_value = int(hashlib.md5(str(app_name).encode()).hexdigest(), 16)
        return countries[hash_value % len(countries)]

    df["country"] = df["app"].apply(assign_country)

    # --------------------------------------------------
    # Top 5 categories by total installs
    # --------------------------------------------------
    top_5_categories = (
        df.groupby("category")["installs"]
        .sum()
        .sort_values(ascending=False)
        .head(5)
        .index
    )

    df = df[df["category"].isin(top_5_categories)]

    # --------------------------------------------------
    # Aggregate installs by Country & Category
    # --------------------------------------------------
    agg_df = (
        df.groupby(["country", "category"], as_index=False)
        .agg({"installs": "sum"})
    )

    # --------------------------------------------------
    # Create Choropleth Map
    # --------------------------------------------------
    fig = px.choropleth(
        agg_df,
        locations="country",
        locationmode="country names",
        color="installs",
        hover_name="category",
        hover_data={"installs": True},
        color_continuous_scale="Plasma",
        title="Global App Installs (Top 5 Categories)"
    )

    # --------------------------------------------------
    # Highlight installs > 1 Million
    # --------------------------------------------------
    highlight_df = agg_df[agg_df["installs"] > 1_000_000]

    fig.add_scattergeo(
        locations=highlight_df["country"],
        locationmode="country names",
        text=highlight_df["category"],
        mode="markers",
        marker=dict(size=12, color="red"),
        name="Installs > 1M"
    )

    fig.show()

    print("✅ Dashboard visible because current time is within 6–8 PM IST")
